In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, to_date, hour, dayofweek, month
from pyspark.sql.types import *
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import pandas as pd
import joblib
import os

In [2]:
# create sparksession 
spark = SparkSession.builder \
    .appName("FlightDelayTraining") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/18 03:43:12 WARN Utils: Your hostname, refat, resolves to a loopback address: 127.0.1.1; using 10.0.2.15 instead (on interface enp0s3)
26/04/18 03:43:12 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/18 03:43:15 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
#  load  data 
BASE_PATH = "/home/ahmed-refat/Desktop/flights & airports/Raw Data(batch)"

flights_df = spark.read.csv(
    f"{BASE_PATH}/NYC_delays.csv",
    header=True,
    inferSchema=True
)


weather_df = spark.read.csv(
    f"{BASE_PATH}/weather.csv",
    header=True,
    inferSchema=True
)

airports_df = spark.read.csv(
    f"{BASE_PATH}/World_Airports.csv",
    header=True,
    inferSchema=True
)

print(f"Flights: {flights_df.count()} rows")
print(f"Weather: {weather_df.count()} rows")
print(f"Airports: {airports_df.count()} rows")



Flights: 616823 rows
Weather: 8832 rows
Airports: 75052 rows


In [7]:
#   هبص علي الداتا والاسكيما بتاعت كل داتا ست منهم 


print("=== FLIGHTS ===")
flights_df.printSchema()
flights_df.show(3)

print("=== WEATHER ===")
weather_df.printSchema()
weather_df.show(3)

print("=== AIRPORTS ===")
airports_df.printSchema()
airports_df.show(3)


=== FLIGHTS ===
root
 |-- FL_DATE: string (nullable = true)
 |-- OP_UNIQUE_CARRIER: string (nullable = true)
 |-- OP_CARRIER_FL_NUM: integer (nullable = true)
 |-- ORIGIN_AIRPORT_ID: integer (nullable = true)
 |-- ORIGIN_AIRPORT_SEQ_ID: integer (nullable = true)
 |-- ORIGIN_CITY_MARKET_ID: integer (nullable = true)
 |-- ORIGIN: string (nullable = true)
 |-- ORIGIN_CITY_NAME: string (nullable = true)
 |-- ORIGIN_STATE_NM: string (nullable = true)
 |-- DEST_AIRPORT_ID: integer (nullable = true)
 |-- DEST_AIRPORT_SEQ_ID: integer (nullable = true)
 |-- DEST_CITY_MARKET_ID: integer (nullable = true)
 |-- DEST: string (nullable = true)
 |-- DEST_CITY_NAME: string (nullable = true)
 |-- DEST_STATE_NM: string (nullable = true)
 |-- CRS_DEP_TIME: integer (nullable = true)
 |-- DEP_TIME: double (nullable = true)
 |-- DEP_DELAY: double (nullable = true)
 |-- DEP_DELAY_NEW: double (nullable = true)
 |-- CRS_ARR_TIME: integer (nullable = true)
 |-- ARR_TIME: double (nullable = true)
 |-- ARR_DELAY:

In [9]:
from pyspark.sql.functions import col, to_timestamp, to_date, split, lower, when, hour

#-----------------------------------
# CLEAN FLIGHTS

flights_clean = flights_df.select(
    col("FL_DATE"),
    col("OP_UNIQUE_CARRIER").alias("airline"),
    col("ORIGIN"),
    col("DEST"),
    col("ORIGIN_CITY_NAME"),
    col("CRS_DEP_TIME"),
    col("DEP_DELAY"),
    col("DISTANCE"),
    col("CANCELLED"),
    col("DIVERTED")
) \
.filter(col("CANCELLED") == 0) \
.filter(col("DIVERTED") == 0) \
.filter(col("DEP_DELAY").isNotNull()) \
.filter(col("ORIGIN").isNotNull()) \
.withColumn(
    "date",
    to_date(to_timestamp(col("FL_DATE"), "M/d/yyyy h:mm:ss a"))
) \
.withColumn(
    "dep_hour",
    (col("CRS_DEP_TIME") / 100).cast("int")
) \
.withColumn(
    "city_clean",
    lower(split(col("ORIGIN_CITY_NAME"), ",").getItem(0))
)

print(f"Flights after cleaning: {flights_clean.count()} rows")


#-------------------------------------
# CLEAN WEATHER

weather_clean = weather_df.select(
    to_date(col("datetime")).alias("date"),
    hour(col("datetime")).alias("weather_hour"),
    lower(col("name")).alias("city_clean"),
    col("temp").alias("temperature"),
    col("windspeed").alias("wind_speed"),
    col("windgust").alias("wind_gust"),
    col("precip").alias("precipitation"),
    col("visibility"),
    col("humidity"),
    col("cloudcover"),
    col("severerisk")
) \
.filter(col("temperature").isNotNull()) \
.filter(col("wind_speed").isNotNull())

# fix typo
weather_clean = weather_clean.withColumn(
    "city_clean",
    when(col("city_clean") == "newyourk", "new york")
    .otherwise(col("city_clean"))
)

print(f"Weather after cleaning: {weather_clean.count()} rows")


#------------------------------------------
# CLEAN AIRPORTS

airports_clean = airports_df.select(
    col("iata_code"),
    col("latitude_deg").alias("airport_lat"),
    col("longitude_deg").alias("airport_lon"),
    col("elevation_ft")
) \
.filter(col("iata_code").isNotNull())

print(f"Airports after cleaning: {airports_clean.count()} rows")


#-------------------------------------------------------------
# JOIN 1: FLIGHTS + AIRPORTS

df = flights_clean.join(
    airports_clean,
    flights_clean.ORIGIN == airports_clean.iata_code,
    "left"
).drop("iata_code")


#-------------------------------------------------------------
# JOIN 2: FLIGHTS + WEATHER

df = df.join(
    weather_clean,
    (df.city_clean == weather_clean.city_clean) &
    (df.date == weather_clean.date) &
    (df.dep_hour == weather_clean.weather_hour),
    "left"
)

print(f"After join: {df.count()} rows")
df.show(3)


#---------------------------------------------------------------
# TARGET COLUMN

df = df.withColumn(
    "is_delayed",
    when(col("DEP_DELAY") > 15, 1).otherwise(0)
)


#---------------------------------------------------------------
# FINAL DATASET
final_df = df.select(
    col("airline"),
    col("airport_lat").cast("float"),
    col("airport_lon").cast("float"),
    col("temperature").cast("float"),
    col("wind_speed").cast("float"),
    col("wind_gust").cast("float"),
    col("precipitation").cast("float"),
    col("visibility").cast("float"),
    col("humidity").cast("float"),
    col("cloudcover").cast("float"),
    col("is_delayed")
) \
.dropna()

print(f"Final dataset: {final_df.count()} rows")
final_df.show(3)

Flights after cleaning: 599617 rows
Weather after cleaning: 8832 rows
Airports after cleaning: 8868 rows


After join: 600213 rows


+--------------------+-------+------+----+----------------+------------+---------+--------+---------+--------+----------+--------+-----------+-----------+-----------+------------+----------+------------+----------+-----------+----------+---------+-------------+----------+--------+----------+----------+
|             FL_DATE|airline|ORIGIN|DEST|ORIGIN_CITY_NAME|CRS_DEP_TIME|DEP_DELAY|DISTANCE|CANCELLED|DIVERTED|      date|dep_hour| city_clean|airport_lat|airport_lon|elevation_ft|      date|weather_hour|city_clean|temperature|wind_speed|wind_gust|precipitation|visibility|humidity|cloudcover|severerisk|
+--------------------+-------+------+----+----------------+------------+---------+--------+---------+--------+----------+--------+-----------+-----------+-----------+------------+----------+------------+----------+-----------+----------+---------+-------------+----------+--------+----------+----------+
|2/1/2025 12:00:00 AM|     AA|   JFK| LAX|    New York, NY|         659|     -5.0|  2475

Final dataset: 155188 rows
+-------+-----------+-----------+-----------+----------+---------+-------------+----------+--------+----------+----------+
|airline|airport_lat|airport_lon|temperature|wind_speed|wind_gust|precipitation|visibility|humidity|cloudcover|is_delayed|
+-------+-----------+-----------+-----------+----------+---------+-------------+----------+--------+----------+----------+
|     AA|  40.639446|  -73.77932|       38.7|      11.5|     18.1|          0.0|       9.9|   77.31|     100.0|         0|
|     AA|  40.639446|  -73.77932|       30.1|      12.7|     26.2|          0.0|       9.9|   24.86|       0.0|         0|
|     AA|  40.639446|  -73.77932|       35.6|       9.1|     27.2|          0.0|       9.9|   68.14|      95.7|         1|
+-------+-----------+-----------+-----------+----------+---------+-------------+----------+--------+----------+----------+
only showing top 3 rows


In [10]:
# تحويل لـ pandas
pdf = final_df.toPandas()

# encode الـ airline
pdf["airline"] = pdf["airline"].astype("category").cat.codes

# تقسيم الداتا
X = pdf.drop("is_delayed", axis=1)
y = pdf["is_delayed"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {len(X_train)} | Test: {len(X_test)}")
print(f"Delay ratio: {y.mean():.2%}")

Train: 124150 | Test: 31038
Delay ratio: 22.20%


In [11]:
#  train model

model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=len(y[y==0]) / len(y[y==1]),
    min_child_weight=5,
    early_stopping_rounds=20,
    random_state=42,
    eval_metric="logloss"
)

model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=10
)

print("Training done!")

[0]	validation_0-logloss:0.68999
[10]	validation_0-logloss:0.65946
[20]	validation_0-logloss:0.64292
[30]	validation_0-logloss:0.62967
[40]	validation_0-logloss:0.62084
[50]	validation_0-logloss:0.61336
[60]	validation_0-logloss:0.60897
[70]	validation_0-logloss:0.60627
[80]	validation_0-logloss:0.60298
[90]	validation_0-logloss:0.59978
[100]	validation_0-logloss:0.59741
[110]	validation_0-logloss:0.59477
[120]	validation_0-logloss:0.59236
[130]	validation_0-logloss:0.59011
[140]	validation_0-logloss:0.58790
[150]	validation_0-logloss:0.58571
[160]	validation_0-logloss:0.58431
[170]	validation_0-logloss:0.58246
[180]	validation_0-logloss:0.58079
[190]	validation_0-logloss:0.57940
[200]	validation_0-logloss:0.57801
[210]	validation_0-logloss:0.57649
[220]	validation_0-logloss:0.57527
[230]	validation_0-logloss:0.57437
[240]	validation_0-logloss:0.57325
[250]	validation_0-logloss:0.57193
[260]	validation_0-logloss:0.57111
[270]	validation_0-logloss:0.57035
[280]	validation_0-logloss:0.56

In [12]:
# تقيم المودل
from sklearn.metrics import classification_report

y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred, target_names=["On-Time", "Delayed"]))

              precision    recall  f1-score   support

     On-Time       0.88      0.75      0.81     24146
     Delayed       0.42      0.63      0.50      6892

    accuracy                           0.72     31038
   macro avg       0.65      0.69      0.65     31038
weighted avg       0.77      0.72      0.74     31038



In [14]:
#حفظ المودل 
import joblib, os

MODEL_PATH = "/home/ahmed-refat/Desktop/flights & airports/ML model"
os.makedirs(MODEL_PATH, exist_ok=True)

joblib.dump(model, f"{MODEL_PATH}/xgboost_delay_model.pkl")
joblib.dump(list(X.columns), f"{MODEL_PATH}/feature_names.pkl")

print(f"Model saved!")
print(f"Features: {list(X.columns)}")

Model saved!
Features: ['airline', 'airport_lat', 'airport_lon', 'temperature', 'wind_speed', 'wind_gust', 'precipitation', 'visibility', 'humidity', 'cloudcover']


In [19]:
print(f"Delay ratio: {y.mean():.2%}")


Delay ratio: 22.20%
